SOW-BKI230A Deep Learning<br>Spring 2025

Assignment 2<br>Multilayer Perceptrons

**Name:**

Angelina Podolako

**S-number:**

s1125886

In this assignment, you will be building a multilayer perceptron (MLP) using PyTorch. Specifically, you will be creating an MLP that can classify images of hand-written digits from the MNIST dataset.

The MNIST dataset is a widely-used benchmark dataset in machine learning and computer vision. It consists of a set of 70,000 images of handwritten digits from 0 to 9, with 10,000 images reserved for testing and 60,000 images for training. Each image is a grayscale 28x28 pixel image, and the task is to classify each image into one of the 10 digit classes.

The MNIST dataset has become a standard dataset for testing machine learning algorithms, particularly in the field of image recognition. Its popularity is due in part to its relative simplicity and small size, which make it easy to work with and experiment with different algorithms and models.

Many researchers have used the MNIST dataset as a benchmark for evaluating the performance of various machine learning models, including neural networks, support vector machines, and decision trees. The dataset has also been used as a starting point for many introductory tutorials and courses in machine learning and computer vision.

**Question 1:**
Before proceeding, explain in your own words what an MLP is and its primary components. How does it differ from a single-layer perceptron?

**Answer 1:**

MLP = MultiLayer Perceptron = idealised model of biological neurons, artificial neural network. It consist of multiple layers of artificial neurons, where each neuron from one layer are connected to all neurons from previous and next layers. There are 3 types of layers: input layer, one or mopre hidden layers, output layer.     
Each neuron in hodden layer get input data, and applies non linear transformation to weighted sum of outputs from nodes in previous layers.  

    
The main components of MLP:  
1) Input layer — recieves input data, one node for eah feature(for example, image pixels).  
2) Hidden layers process data by performing calculations and nonlinear transformations. Responsible for transforming input features into those that are more suitable for output layers.  
3) The output layer produces output vector. Generates model predictions -binary/multiclass lassification(in the case of MNIST, the probability of an image belonging to one of 10 classes).  
4) Activation functions — provide non-linearity of the model, allowing them learn and approximate complex functions(for example, ReLU, Sigmoid, Step, Linear, Softmax).  
5) Weights and biases are the parameters that are trained during the training of the model.  

    
The difference between MLP and a single-layer perceptron is that an MLP has one or more hidden layers, whereas a single-layer perceptron consists only of input and output layers. This makes MLP capable of learning complex nonlinear dependencies, whereas a single-layer perceptron can only solve linearly separable problems.


In [10]:
import torch #основная библиотека для работы с тензорами и вычислениями.
import torch.nn as nn # модуль для создания нейронных сетей.
import torch.nn.functional as F #функции активации и потерь
import torchvision.datasets as datasets # содержит наборы данных, в том числе MNIST
import torchvision.transforms as transforms #позволяет применять преобразования к изображениям.
import torch.utils.data as data #инструменты для загрузки данных.

### Step 1: Set seed for reproducibility
This sets the random seed for the PyTorch library to 0. This ensures that the random initialization of the model's parameters, as well as any other random operations, are the same each time the code is run, making the results reproducible.

In [11]:
torch.manual_seed(0)

**Question 2:**
Why is setting a random seed important in machine learning experiments?

**Answer 2:**

The installation of a random seed is important because in machine learning many processes (initialization of weights, data partitioning, augmentation) depend on random numbers. If you do not fix the seed, then the results may vary with each experiment run. This hinders the reproducibility of experiments, so it will be difficult to compare changes, similarity of results, and monitor model changes if it uses absolutely accurate numbers each time.   

### Step 2: Use GPU if available
This checks if a GPU is available and sets the device variable to "cuda" if one is available, otherwise it sets it to "cpu". This allows the code to use GPU acceleration if possible, which can greatly speed up computations.

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

**Question 3:**
Explain the significance of using GPU acceleration in deep learning and how the torch.device function facilitates this in PyTorch.

**Answer 3:**

Deep neural networks contain a lot of parameters, and their training requires a huge amount of calculations. The GPU significantly speeds up the process, as it is optimized for parallel computing.     

In Py Torch, torch.device makes it easy to switch between CPU and GPU, ensuring cross-platform code compatibility.

Code above: Checks if the graphics card (GPU) is available.  
If yes, it sets device = "cuda", if not, it sets device = "cpu"  

### Step 3: Define data transforms
This defines a series of data transforms to be applied to the MNIST dataset. The ToTensor() transform converts the data to a PyTorch tensor, and the Normalize() transform normalizes the data by subtracting the mean and dividing by the standard deviation. The values (0.1307,) and (0.3081,) are the mean and standard deviation of the MNIST dataset.

In [13]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

* transforms.Compose([...]) — combines several transformations.
* transforms.ToTensor() — converts an image to a PyTorch tensor.
* transforms.Normalize((0.1307,), (0.3081,)) — normalizes image pixels using the average and standard deviation of the MNIST dataset

**Question 4:**
Why is data normalization important in training neural networks? How do the values 0.1307 and 0.3081 relate to the MNIST dataset?

**Answer 4:**

Normalization is important because it
* accelerates model learning – the neural network finds optimal parameters faster
* makes the input data more stable – the numbers are neither too small nor too large
* helps to avoid problems with gradient descent. Without normalization, the model may converge slowly or fail to learn at all.


 The values 0.1307 and 0.3081 are constants of the average and standard deviation of pixels in all MNIST images. They help to bring the data to a standard form, improving the training of the model.

* 0.1307 is the average pixel brightness value in all MNIST images (that is, on average, pixels are neither pure black nor pure white, but slightly gray).
* 0.3081 is the standard deviation, which shows how much the pixel values deviate from the average (the greater the deviation, the greater the spread of values).
     
    
When normalizing, we subtract the average from each pixel and divide it by the standard deviation.:  
*Xnorm=  (X − 0.1307) / (0.3081)*  
This brings all pixel values to a range of about -1 to 1, which is convenient for neural networks.  

Without normalization, the model will still work, but training will be slower and worse, especially if the input data has a large range of values.

### Step 4: Load MNIST dataset
This loads the MNIST dataset from the internet and applies the previously defined transform to it. The dataset is divided into training and testing sets using the train argument.

In [14]:
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

* train_data – contains training data (60,000 images).
* test_data – contains test data (10,000 images).
* train=True - training sample is loaded, and train=False – the test sample.
* download=True downloads files if they are not available locally.
* transform=transform specifies which transformations to apply to the images.

### Step 5: Create data loaders
This creates data loaders for the training and testing sets. The DataLoader class loads the data in batches and shuffles it if the shuffle argument is set to True. The batch_size argument specifies the number of images in each batch, and num_workers sets the number of parallel data loading processes.

In [15]:
train_loader = data.DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
test_loader = data.DataLoader(test_data, batch_size=64, shuffle=False, num_workers=2)

Data loaders help you load data in batches, which speeds up model training.
* batch_size=64 – the number of images in one batch.
* shuffle=True – shuffles the data in the training set. This is important so that the model does not memorize the order of the examples and learns more efficiently.
* shuffle=False (test set) – no need to shuffle, as the test data is simply evaluated, not trained.
* num_workers=2 – the number of processes downloading data in parallel (speeds up the work).

**Question 5:**
Discuss the importance of shuffling the training data and why it is not necessary to shuffle the test data.

**Answer 5:**

Shuffle (shuffle=True) in training helps to avoid memorizing the order of the data and makes Gradient Descent more stable. If we hadn't shuffled the data, the model could have memorized the sequence of examples rather than identifying patterns. Then it will be biased model, which has been good in memorizing, and not predicting, so it will be overfitted.

 There is no need to shuffle (shuffle=False) the test data, because testing is just an evaluation of the model. Mixing here does not affect the quality of the predictions. It does not learn from test data!

### Step 6: Define MLP model
This defines the MLP model using the nn.Module class in PyTorch. The MLP model consists of three fully connected (linear) layers, which are defined in the constructor (__init__ method) of the MLP class. The nn.Linear function creates a linear transformation of the input tensor x to the output tensor y using the equation:

$$y = xA^⊤ + b$$

where A is a weight matrix, b is a bias vector, and T denotes matrix transpose. The nn.Linear function takes two arguments: the input size and the output size of the linear layer. For example, in the line:

$$\text{self.fc1 = nn.Linear(784, 128)}$$

the first fully connected layer fc1 takes an input tensor of size 784 and produces an output tensor of size 128.

The super(MLP, self).__init__() line calls the constructor of the parent class nn.Module. This is necessary for the model to inherit all of the properties and methods of the nn.Module class.

The forward method of the MLP class defines the forward pass of the model. The x.view(-1, 784) line reshapes the input tensor x into a 2D tensor of shape (batch_size, 784). This is necessary because the input images are 28x28 pixels, and the MLP model expects a 1D input of size 784. The -1 argument in view indicates that the size of that dimension should be inferred automatically based on the size of the other dimensions and the number of elements in the tensor.

The F.relu function applies the Rectified Linear Unit (ReLU) activation function to the output of the first two fully connected layers, fc1 and fc2. ReLU activation is a non-linear activation function that is commonly used in neural networks to introduce non-linearity into the model. It has the form:

$$f(x) = \max(0, x)$$

where x is the input to the activation function.

The final fully connected layer fc3 has no activation function applied to its output, which means that it is simply a linear transformation of the input. The output of the model is returned from the forward method as x.

Typically, in classification problems where we want to predict the probabilities of each class, we use the softmax activation function on the output layer of the neural network to convert the raw output values into a probability distribution.

However, we are not using the softmax activation function on the output layer because we are using the nn.CrossEntropyLoss() loss function, which combines the softmax function and the negative log-likelihood loss function into a single function. The softmax function is used to convert the raw output values into a probability distribution, and the negative log-likelihood loss function is used to measure the difference between the predicted probability distribution and the true probability distribution.

The nn.CrossEntropyLoss() loss function expects raw output values from the final layer of the MLP model, rather than the probabilities obtained from applying softmax activation. The raw output values are passed through the loss function, which internally applies the softmax function and calculates the cross-entropy loss.

Therefore, we don't need to apply the softmax function manually in the MLP model defined in step 6. The nn.CrossEntropyLoss() loss function takes care of applying the softmax function for us.

In summary, the MLP model defined in step 6 takes an input tensor of size (batch_size, 1, 28, 28) representing a batch of images, and produces an output tensor of size (batch_size, 10) representing the predicted class probabilities for each image in the batch. The model consists of three fully connected layers with ReLU activation functions applied to the outputs of the first two layers.

In [16]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__() #MLP из 3 полностью связанных (линейных) слоев-Лин функц - линейное преобразование вход тензора x в выход тензор y, используя уравнение: y=xA⊤+b
        self.fc1 = nn.Linear(784, 128) # Первый слой: 784 входа (развёрнутое изображение 28x28) → 128 выхода (нейронов)
        self.fc2 = nn.Linear(128, 128) # Второй слой: 128 входов → 128 выхода
        self.fc3 = nn.Linear(128, 10) # Третий слой: 128 входов → 10 выхода (классы 0-9) (выдаёт 10 чисел (одно для каждого класса цифр от 0 до 9))

    def forward(self, x):
        x = x.view(-1, 784) # Преобразуем 28x28 в вектор длиной 784 # Входные данные
        x = F.relu(self.fc1(x))   #Данные проходят через первый слой (fc1), а затем применяется функция активации ReLU которая делает все отрицательные значения равными нулю. Это добавляет нелинейность в модель
        x = F.relu(self.fc2(x)) # Второй слой + ReLU #Данные проходят через второй слой (fc2) и снова применяется функция ReLU.
        x = self.fc3(x) # Третий слой (fc3) (выходной, без активации)   Это выходной слой, который возвращает 10 значений, соответствующих вероятностям принадлежности к каждому из 10 классов.
        return x

**Question 6:**
Using the model definition above, trace how an input batch of MNIST images is transformed through the MLP. For an input batch size of 64, explain the shape of the data at each step of processing through the model's layers.

**Answer 6:**

A batch of 64 images (batch_size = 64).

1) The original size: (64, 1, 28, 28)
* 64 – package size,
* 1 – black and white channel,
* 28x28 is the size of the image.

2) After x.view(-1, 784): (64, 784)  
* Expand the image into a vector of length 784(28*28).
* -1 means the size of dimension automatically based on the size of the other dimensions and the number of elements in the tensor.

3) After self.fc1(x): (64, 128)  
* The **first fully connected layer** reduces the dimension to 128.

4) After F.relu(self.fc1(x)): (64, 128)  
* The size remains the same, but the *values are now non-linearly transformed by the Relu function.*

5) After self.fc2(x): (64, 128)  
* **The second fully connected layer** retains a size of 128.

6) After F.relu(self.fc2(x)): (64, 128)  
* The size remains the same, but the *values are now non-linearly transformed by the Relu function.*

7) After self.fc3(x): (64, 10)  
* **The output layer** outputs 10 values, one for each class (digits from 0 to 9).

**Question 7:**
In the model definition above, the activation function after each linear layer is ReLU. Replace the ReLU activation after the first linear layer with a Sigmoid activation, and adjust the forward method accordingly.

In [17]:
# Answer 7:
def forward(self, x):
    x = x.view(-1, 784)
    x = torch.sigmoid(self.fc1(x))  # after the first linear layer(fc1) - a Sigmoid activation
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

**Question 8:**
The current model architecture has two hidden layers. Add a third hidden layer to the network with 64 neurons and apply a ReLU activation to this layer. Explain how you would expect this change to influence the model's capacity.

In [18]:
# Answer 8:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 64) #Inp 128 neurons, output 64
        self.fc4 = nn.Linear(64, 10)  #output layer= 64 neurons into 10 classes

    def forward(self, x):
        x = x.view(-1, 784)
        x = torch.sigmoid(self.fc1(x))  # after the first linear layer(fc1) - a Sigmoid activation
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

* Pros: the expressive power of the model will increase, it will be better able to find complex patterns.  
 - Adding a third layer increases the number of parameters of the model, making it more complex and capable of learning more complex functions.  
 - The model now has more layers for extracting and combining features from the input data.  
 - With an additional layer, the model can better capture complex dependencies in the data, which can improve its performance on complex tasks.
   
* Cons: the training time will increase, and the model may start overfitting if regularization is not used.
 - However, if there is not enough data, this can lead to overfitting (the model will remember the data instead of generalizing)
  - As the number of layers and parameters increases, the model becomes more prone to overfitting, especially if there is little data.
  - To avoid this, regularization methods such as Dropout or L1 or L2 regularization can be used.

### Step 7: Initialize model, loss function, and optimizer
This initializes the MLP model, loss function, and optimizer. The to method is used to move the model to the device specified in the device variable. The CrossEntropyLoss() function is used as the loss function, and the SGD optimizer is used with a learning rate of 0.01. The optimizer is used to update the model's parameters during backpropagation.

In [24]:
model = MLP().to(device) #используется для перемещения модели на устройство, указанное в переменной device
criterion = nn.CrossEntropyLoss() #применяется для многоклассовой классификации(loss func)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01) #оптимизатор SGD (стохастический градиентный спуск) обновляет параметры модели во время обратного распространения ошибки с шагом обучения 0.01.

### Step 8: Train the model
This trains the MLP model for 10 epochs using the training data loader. The outer loop iterates over the number of epochs, and the inner loop iterates over the data in batches. For each batch, the data is moved to the device specified in the device variable, the optimizer's gradients are reset to zero using optimizer.zero_grad(), forward propagation is performed using model(images), the loss is calculated using the criterion function, and backward propagation is performed using loss.backward(). The optimizer's step() method updates the model's parameters using the calculated gradients. The running loss is accumulated over each batch, and the average loss is printed at the end of each epoch.

In [20]:
for epoch in range(10): #Внешний цикл проходит по количеству эпох 1Эпоха — это один полный проход по всем данным в обучающем наборе
    model.train() #b режиме обучения модель активирует такие функции, как Dropout и Batch Normalization
    running_loss = 0.0 #для суммирования потерь (ошибок) на каждом батче данных в течение одной эпохи

    for images, labels in train_loader: # внутренний — по батчам данных. #train_loader — это DataLoader, который загружает данные батчами (пакетами). #один батч изображений (images) и соответствующих меток (labels
        images = images.to(device) #Данные переносятся на устройство (device)
        labels = labels.to(device) #Данные перемещаются на устройство (CPU или GPU), на котором будет происходить обучение модели
        optimizer.zero_grad() #Градиенты оптимизатора сбрасываются
        outputs = model(images) #forward propagation is performed using model(images)  на выходе получаются предсказания (outputs
        loss = criterion(outputs, labels)  #loss is calculated using the criterion function сравнивает предсказания модели (outputs) с истинными метками (labels) и вычисляет ошибку
        loss.backward() # backward propagation is performed  Вычисляются градиенты функции потерь по всем параметрам модели
        optimizer.step() #optimizer's step() method updates the model's parameters using the calculated gradients
        running_loss += loss.item() #running loss is accumulatedНакопленный over each batch, and the average loss средний убыток is printed at the end of each epoch.

    print('Epoch {} training loss: {:.3f}'.format(epoch+1, running_loss/len(train_loader)))

Epoch 1 training loss: 2.160
Epoch 2 training loss: 1.073
Epoch 3 training loss: 0.549
Epoch 4 training loss: 0.405
Epoch 5 training loss: 0.347
Epoch 6 training loss: 0.315
Epoch 7 training loss: 0.291
Epoch 8 training loss: 0.270
Epoch 9 training loss: 0.251
Epoch 10 training loss: 0.234


**Question 9:**
Describe the process of backpropagation in the training loop above and mention why optimizer.zero_grad() is necessary.

**Answer 9:**

1. Backpropagation is an algorithm for updating the weights of a neural network based on gradient descent.  
Backpropagation is the process by which a model "learns" from its mistakes. It consists of two main steps:
* Forward Pass:
 - Data (such as images) is passed through the model.
 - The model makes predictions.
 - The loss function (error) is calculated, which shows how much the model's predictions differ from the correct answers.  
(outputs = model(images)  # model make predictions  
loss = criterion(outputs, labels)  # Computes loss(error))
  
* Reverse Pass:
 - The model calculates how each of its parameters (weights) affects the error.
 - To do this, gradients (derivatives of the loss function for each parameter) are calculated.
 - Gradients show which way and by how much the parameters need to be changed in order to reduce the error.  
(loss.backward() #calculates the gradient of the loss function with respect to the model parameters.    
optimizer.step() #updates the parameters based on these gradients)

    
2. Gradients accumulate with each loss call.backward(). To avoid accumulating them, they need to be zeroed before each optimization step.
* Gradients are accumulating:
 - When call loss.backward(), gradients are calculated and added to those that are already in the model parameters.
 - If do not reset the gradients before the next step, they will accumulate, which will lead to incorrect parameter updates.
 (optimizer.zero_grad()  #reset gradients before each step  
 This line ensures that the gradients are "reset" before calculating new ones. This is important for the correct training of the model.)

### Step 9: Test the model
This tests the MLP model using the testing data loader. The model is set to evaluation mode using model.eval(), and the torch.no_grad() context is used to disable gradient calculation since the testing is done only for evaluation purposes. The predicted tensor is calculated by taking the maximum value from the outputs tensor along the second dimension. The total number of images and the number of correctly predicted images are calculated and used to calculate the test accuracy, which is printed at the end.

In [21]:
model.eval() #Устанавливается режим оценки что отключает такие механизмы, как dropout и batch normalization.
correct = 0
total = 0

with torch.no_grad(): #отключающий вычисление градиентов, так как во время тестирования обратное распространение ошибки не требуется.
    for images, labels in test_loader: #Прогоняем данные через модель.
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1) #тобы получить предсказанные метки. прогнозируемый тензор вычисляется- макс значения из выходного тензора по второму измерению.
        total += labels.size(0)
        correct += (predicted == labels).sum().item() #количество верных предсказаний

test_accuracy = 100 * correct / total #и рассчитываем точность (accuracy)
print('Accuracy on test data: {:.2f}%'.format(test_accuracy))

Accuracy on test data: 93.45%


**Question 10:**
Explain why it's necessary to use model.eval() and torch.no_grad() during evaluation.

**Answer 10:**

* model.val()
 - Switches the model to evaluation mode, disabling Dropout and Batch Normalization.
 - Provides correct predictions based on test data.

* torch.no_grade()  
 - Disables the calculation of gradients, which saves memory and speeds up execution. (since gradients are not needed in the test process.)
 - !use it during testing, as back propagation is not required.

### Step 10: Save the model
This saves the MLP model's state dictionary to a file called mlp_model.pt using the torch.save function.

In [22]:
torch.save(model.state_dict(), 'mlp_model.pt') #Сохранение параметров модели (весов) в файл mlp_model.pt с помощью torch.save()

### Step 11: Load the saved model
This loads the saved MLP model's state dictionary from the file mlp_model.pt and initializes a new instance of the MLP class using the MLP() constructor. The state dictionary is loaded into the model using the load_state_dict method. The loaded model is then ready for use.

In [23]:
model = MLP().to(device) #оздается новая модель MLP()
model.load_state_dict(torch.load('mlp_model.pt')) # в нее загружаются сохраненные параметры (load_state_dict).  позволяет повторно использовать модель без необходимости повторного обучения.

<ipython-input-23-34a2f4838f1e>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('mlp_model.pt'))


<All keys matched successfully>

**Question 11:**
Discuss the significance of saving and loading model parameters for deep learning models.

**Answer 11:**

* Saving the model allows you to avoid re-learning and use an already trained model in the future.  
1) Continuing education
 - Deep model training can take a long time (hours, days, or even weeks).
 - If the learning process is interrupted (for example, due to equipment failure or lack of time), the saved parameters allow you to continue learning from where it left off, instead of starting over.

2) Using a trained model
 - After training, the model can be used to make predictions based on new data.
 - Saving the parameters allows you to "fix" the trained model and use it at any time without the need for repeated training.

3) Exchange of models
 - Saved parameters can be shared with other developers or used on other devices.
 - This is especially useful in teamwork or when deploying a model in production.

4) Saving resources
 - Model training requires significant computing resources (GPU/CPU, time, electricity).
 - Saving the parameters avoids re-learning, which saves resources.


* Loading the model helps:
 - Deploy models in production.
 - Do further training (fine-tuning) on new data.
 - Do transfer learning.

1) Continuing education
 - If the training was interrupted, you can download the saved parameters and continue the training from the same location.

2) Using a trained model
 - The loaded parameters allow you to use the model for predictions without having to retrain it.

3) Transfer training
 - You can load the parameters of a pre-trained model (for example, on a large dataset) and retrain it on your own data.
 - This is especially useful when you have little training data.

***Thus, saving and loading model parameters saves computing resources and speeds up development.***